# BPT-MOTUS (vs. other methods) Example

# Load Data
## Load data

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=1
!export BART_CPU_ONLY=1
!export CUDA_LAUNCH_BLOCKING=1

import sys
import os

os.environ['BART_TOOLBOX_PATH'] = '/home/rinbha/Packages/bart-0.9.00'
os.environ['BART_CPU_ONLY'] = '1'
sys.path.append(os.path.join(os.environ['BART_TOOLBOX_PATH'], 'python'))

sys.path.insert(0, "/home/rinbha/Research/BPT/BPT_MOTUS/nonrigid-motion-bpt")
sys.path.insert(0, "/home/rinbha/Research/BPT/BPT_MOTUS/bpt_mrmotus_2025/bpt_mrmotus_2025/mrmotus_custom_packages/torch-interpol")

import matplotlib.pyplot as plt
# plotting
plt.rcParams.update({
    'axes.spines.top': False,           # Remove top spine
    'axes.spines.right': False,         # Remove right spine
    'axes.titlesize': 20,               # Increase title font size
    'axes.labelsize': 18,               # Increase axis label font size
    'xtick.labelsize': 16,              # Increase x-tick label size
    'ytick.labelsize': 16,              # Increase y-tick label size
    "text.usetex": False,                # Enable LaTeX 
    'legend.fontsize': 12,
    'font.family': 'Avenir' # Avenir font
})

In [ ]:
from bpt_motus.io import RadialArchive

# Point to raw radial scan target
base_dir = "/mikLKS/rinbha/BPT/MRMOTUS/example_data/bpt_motus_clean_before_split"  # sandboxed copy (symlinked raw inputs), isolated from the original bpt_motus/ data
inpdir = f"{base_dir}/hires_ute" 
radial = RadialArchive(inp_dir=inpdir)
# radial.get_ksp(force_reload=False)

# print(f"xk_time: {radial.xk_time.shape}")
# print(f"coords_time: {radial.coords_time.shape}")
# print(f"dcf_time: {radial.dcf_time.shape}")

## Note: reordered processing (clean/compress before phase-splitting)

This notebook is a variant of `bpt_motus_example.ipynb` that runs `SplitXkBPT` and
`ProcessBPT` **once**, on the raw continuous acquisition, *before* `SplitRadialAcq`
splits it into the `no_motion`/`calib`/`inf` phases -- rather than running them
independently three times, once per already-split phase. `SplitRadialAcq` then slices
the already-cleaned/compressed `xk_cleaned_comp.npy` and already-processed
`bpts_proc.npy` the same way it slices the raw data, so every phase directory ends up
with the same filenames as before.

This guarantees a single, consistent coil-compression basis and BPT-tone
peak/offset estimate across all three phases (rather than relying on manually
passing `pca_fname` between per-phase runs), at the cost of fitting the BPT-PCA
basis over the whole recording instead of just the calibration window.
Everything from `NoMotionReference` onward is unchanged.


## Split k-space and BPT (on the raw, pre-split acquisition)

In [ ]:
from bpt_motus.preprocessing import SplitXkBPT

# Run once on the full, continuous, pre-split hires acquisition (fits + saves the coil-compression PCA)
splitter_hr = SplitXkBPT(inp_dir=inpdir, verbose=True)
splitter_hr.run(force_reload=True)

# If calibration is drawn from a separate lowres acquisition, mirror this call on lowres_dir,
# passing pca_fname=splitter_hr.pca_fname to reuse the same coil-compression basis:
# lowres_dir = f"{base_dir}/lowres_ute"
# splitter_lr = SplitXkBPT(inp_dir=lowres_dir, pca_fname=splitter_hr.pca_fname, verbose=True)
# splitter_lr.run(force_reload=True)


## Process B+PT signals (on the raw, pre-split acquisition)

In [ ]:
from bpt_motus.preprocessing import ProcessBPT
import os

# Run once, globally -- there's now only one BPT-PCA fit for the whole recording, so
# phase="calib" here just means "fit fresh"; there's no more phase="inf" transform-only
# call, since there's no longer a second, independently-processed dataset to transform.
# bpts_pca_fname is passed explicitly since the class's default path assumes a sibling
# "calib" folder (true for a post-split calib_dir/inf_dir, not for a raw hires_dir).
proc_hr = ProcessBPT(inp_dir=inpdir, phase="calib",
                      bpts_pca_fname=os.path.join(inpdir, "bpts_pca.pkl"), verbose=True)
proc_hr.run(force_reload=True)

# Mirror for lowres_dir if calib_source == "lowres" (reusing the same bpts_pca_fname
# to keep calibration's BPT-PCA subspace consistent with whatever the model is trained against):
# proc_lr = ProcessBPT(inp_dir=lowres_dir, phase="calib",
#                       bpts_pca_fname=proc_hr.bpts_pca_fname, verbose=True)
# proc_lr.run(force_reload=True)


## Organize radial scan into phases

Now that the whole recording is cleaned, compressed, and BPT-processed, split it into the three phases -- `SplitRadialAcq` will also slice `xk_cleaned_comp.npy`/`bpts_proc.npy` (found above) into each phase directory automatically.

In [ ]:
from bpt_motus.io import SplitRadialAcq

split_exp = SplitRadialAcq(
    inp_dir=base_dir,
    calib_source="hires",
    no_motion_range=(0, 15000),
    calib_range=(20000, 35000),
    inf_range=(35000, None),
    verbose=True
)
split_exp.run(force_reload=False)

## Process no-motion data

In [ ]:
from bpt_motus.preprocessing import NoMotionReference

nomotion_ref = NoMotionReference(inp_dir=split_exp.no_motion_dir, verbose=True, crop_factor=3, center_out=False)
nomotion_ref.run(force_reload=True)

print(f"Reference Image shape: {nomotion_ref.S.shape}")
print(f"CSM shape: {nomotion_ref.csm.shape}")

## Process motion data

In [ ]:
from bpt_motus.preprocessing import MotionFrames

mf_c = MotionFrames(inp_dir=split_exp.calib_dir, verbose=True)
mf_c.run(force_reload=True)

mf_i = MotionFrames(inp_dir=split_exp.inf_dir, verbose=True)
mf_i.run(force_reload=True)

# Assertion check verifying temporal sizes (Nframes) match perfectly
assert mf_c.xk_frames.shape[1] == mf_c.bpts_frames.shape[0], "Calibration temporal sizes do not match!"
assert mf_i.xk_frames.shape[1] == mf_i.bpts_frames.shape[0], "Inference temporal sizes do not match!"
print("Motion frames and BPT temporal sizes match perfectly.")

# Learn motion fields
## Run optimization on calibration data

In [ ]:
from bpt_motus.viz import BPTVisualizer
bv = BPTVisualizer(bpts=optimizer.bpt_frames)
bv.plot_bpts()

In [ ]:
from bpt_motus.motion import MotionFieldOptimizer

optimizer = MotionFieldOptimizer(
    calib_inpdir=mf_c.save_dir,
    nomotion_inpdir=nomotion_ref.save_dir,
    mode='mrmotus',
    xyz_downsampling=[4, 8, 16],
    t_downsampling=[2,5],
    max_t_init=0.5,
    epochs=20,
    batch_size=50,
    learning_rate=5e-2,
    device='cuda',
    verbose=True,
    force_reload=True
)
optimizer.optimize()

In [ ]:
np.linalg.norm(nomotion_ref.S)

In [ ]:
import torch

frame_id = 10  # pick whichever frame you want

# scaling_per_frame isn't cached across reloads -- recompute it (deterministic, zero-motion based)
if optimizer.scaling_per_frame is None:
    scaling = optimizer._compute_scaling_per_frame()
    optimizer.scaling_per_frame = torch.from_numpy(scaling).to(optimizer.device).to(torch.complex64)

coords_frame = torch.from_numpy(optimizer.coords_frames[frame_id]).to(optimizer.device).to(torch.float32).unsqueeze(0)

with torch.no_grad():
    mf_frame = optimizer.motion_model.forward([frame_id])      # motion field for this frame
    S_warped = optimizer._warp_img(mf_frame)                    # reference warped by that field
    k_pred = optimizer._forward_model(S_warped, coords_frame) * optimizer.scaling_per_frame[frame_id]

xk_actual = torch.from_numpy(optimizer.xk_frames[:, frame_id]).to(optimizer.device).to(torch.complex64).unsqueeze(0)

# both are shape (1, ncoils, spokes, samples)
k_pred = k_pred.squeeze(0).cpu().numpy()       # -> (ncoils, spokes, samples)
xk_actual = xk_actual.squeeze(0).cpu().numpy() # -> (ncoils, spokes, samples)

In [ ]:
scaling[frame_id]

In [ ]:
pl.ImagePlot(k_pred[:,:100,50:-50], colormap="gray")

In [ ]:
pl.ImagePlot(xk_actual[:,:100,50:-50], colormap="gray")

In [ ]:
import numpy as np
pl.LinePlot(np.array(optimizer.dc_loss_log), title="Data Consistency Loss over Epochs")

# Visualize Motion
## Reconstruct calibration motion fields

In [ ]:
optimizer.out_dir, os.path.dirname(optimizer.out_dir)

In [ ]:
from bpt_motus.recon import MotionFieldWarp

calib_warp = MotionFieldWarp(
    model_dir = os.path.join(os.path.dirname(optimizer.out_dir), 'bpt_motus'), # optimizer.out_dir,
    ref_dir = nomotion_ref.save_dir, 
    out_dir = None, 
    bpts_dir = split_exp.calib_dir,
    force_reload = True,
)
calib_warp.run()

In [ ]:
calib_warp.clear_data()

## Reconstruct inference motion fields

In [ ]:
from bpt_motus.recon import MotionFieldWarp

# Reuse the calibration-trained model + no-motion reference; only bpts_dir changes to inference frames
model_dir = os.path.join(os.path.dirname(optimizer.out_dir), 'bpt_motus')
nomotion_dir = nomotion_ref.save_dir
inf_dir = split_exp.inf_dir

calib_warp = MotionFieldWarp(
    model_dir = model_dir,
    ref_dir = nomotion_dir, 
    out_dir = None, 
    bpts_dir = inf_dir,
    phase = 'inf',
)
calib_warp.run()

In [ ]:
calib_warp.warped_frames.shape, calib_warp.full_motion_field.shape

## Plot warped images and motion fields

In [ ]:
from bpt_motus.viz import ImageQuiverPlot
iqp = ImageQuiverPlot(calib_warp.warped_frames.detach().cpu().numpy(), calib_warp.full_motion_field.detach().cpu().numpy(), step=10, scale=50, apply_mask=True, mask_thresh=0.01)

# Error Evaluation
## Run PICS reconstructions

In [ ]:
from bpt_motus.recon import PICSRecon

pics_c = PICSRecon(inp_dir=calib_dir, verbose=True)
pics_c.run()

pics_i = PICSRecon(inp_dir=inf_dir, verbose=True)
pics_i.run()

## Apply forward warp to PICS

In [ ]:
# Apply the learned motion fields to warp the baseline PICS reconstructions
# warp_pics = MotionFieldWarp(model_dir=calib_dir, recon_dir=calib_dir)
# warp_pics.run(apply_to_pics=True)

## Voxel-wise Registration

In [ ]:
from bpt_motus.recon import ElastixRegistration

# Perform registration on both calibration and inference datasets using PICS as the reference grid
reg_c = ElastixRegistration(inp_dir=calib_dir, reference='pics', verbose=True)
reg_c.run()

reg_i = ElastixRegistration(inp_dir=inf_dir, reference='pics', verbose=True)
reg_i.run()

## Error Metrics Evaluation

In [ ]:
from bpt_motus.recon import ErrorEvaluator

# Compute and plot alignment errors, outputting voxel-wise RMSE and rigid transform params
evaluator = ErrorEvaluator(calib_dir=calib_dir, inf_dir=inf_dir, verbose=True)
evaluator.run()
evaluator.plot_metrics()